In [ ]:
from collections import Counter
import numpy as np
import os
import pandas as pd
from sklearn.linear_model import LogisticRegression
import torch.nn as nn

from torch_rnn_classifier import TorchRNNClassifier
from torch_tree_nn import TorchTreeNN
import sst
import utils
%load_ext autoreload
%autoreload 2

In [ ]:
SST_HOME = os.path.join('data', 'sentiment')

## Train set

Our primary train set is the SST-3 train set:

In [ ]:
sst_train = sst.train_reader(SST_HOME)

In [ ]:
sst_train.shape[0]

8544

This is the train set we will use for all the regular homework questions. You are welcome to bring in new datasets for your original system. You are also free to add `include_subtrees=True`. This is very likely to lead to better systems, but it substantially increases the overall size of the dataset (from 8,544 examples to 159,274), which will in turn substantially increase the time it takes to run experiments.

See [this notebook](sst_01_overview.ipynb) for additional details of this dataset.

## Dev sets

We have two development set. SST3-dev consists of sentences from movie reviews, just like SST-3 train:

In [ ]:
sst_dev = sst.dev_reader(SST_HOME)


Our new bakeoff dev set consists of sentences from restaurant reviews:

In [ ]:
bakeoff_dev = sst.bakeoff_dev_reader(SST_HOME)


In [ ]:
bakeoff_dev.sample(3, random_state=1).to_dict(orient='records')


[{'example_id': 57,
  'sentence': 'I would recommend that you make reservations in advance.',
  'label': 'neutral',
  'is_subtree': 0},
 {'example_id': 590,
  'sentence': 'We were welcomed warmly.',
  'label': 'positive',
  'is_subtree': 0},
 {'example_id': 1968,
  'sentence': 'We have been to Oceanaire twice in the last 6 weeks.',
  'label': 'neutral',
  'is_subtree': 0}]

Here is the label distribution:

In [ ]:
bakeoff_dev.label.value_counts()


neutral     1019
positive     777
negative     565
Name: label, dtype: int64

The label distribution for the corresponding test set is similar to this.

## A softmax baseline

This example is here mainly as a reminder of how to use our experimental framework with linear models:

In [ ]:
def unigrams_phi(text):
    return Counter(text.split())



Thin wrapper around `LogisticRegression` for the sake of `sst.experiment`:

In [ ]:
def fit_softmax_classifier(X, y):
    mod = LogisticRegression(
        fit_intercept=True,
        solver='liblinear',
        multi_class='ovr')
    mod.fit(X, y)
    return mod


The experimental run with some notes:

In [ ]:
softmax_experiment = sst.experiment(
    sst.train_reader(SST_HOME),   # Train on any data you like except SST-3 test!
    unigrams_phi,                 # Free to write your own!
    fit_softmax_classifier,       # Free to write your own!
    assess_dataframes=[sst_dev, bakeoff_dev]) # Free to change this during development!

Assessment dataset 1
              precision    recall  f1-score   support

    negative      0.628     0.689     0.657       428
     neutral      0.343     0.153     0.211       229
    positive      0.629     0.750     0.684       444

    accuracy                          0.602      1101
   macro avg      0.533     0.531     0.518      1101
weighted avg      0.569     0.602     0.575      1101

Assessment dataset 2
              precision    recall  f1-score   support

    negative      0.272     0.690     0.390       565
     neutral      0.429     0.113     0.179      1019
    positive      0.409     0.346     0.375       777

    accuracy                          0.328      2361
   macro avg      0.370     0.383     0.315      2361
weighted avg      0.385     0.328     0.294      2361

Mean of macro-F1 scores: 0.416


`softmax_experiment` contains a lot of information that you can use for error analysis; see [this section below](#Error-analysis) for starter code.

## RNNClassifier wrapper

This section illustrates how to use `sst.experiment` with `TorchRNNClassifier`.

To featurize examples for an RNN, we can just get the words in order, letting the model take care of mapping them into an embedding space.

In [ ]:
def rnn_phi(text):
    return text.split()

In [ ]:
def fit_rnn_classifier(X, y):
    sst_glove_vocab = utils.get_vocab(X, mincount=2)
    mod = TorchRNNClassifier(
        sst_glove_vocab,
        early_stopping=True)
    mod.fit(X, y)
    return mod

In [ ]:
rnn_experiment = sst.experiment(
    sst.train_reader(SST_HOME),
    rnn_phi,
    fit_rnn_classifier,
    vectorize=False,  # For deep learning, use `vectorize=False`.
    assess_dataframes=[sst_dev, bakeoff_dev])

Stopping after epoch 58. Validation score did not improve by tol=1e-05 for more than 10 epochs. Final error is 0.36771461740136147

Assessment dataset 1
              precision    recall  f1-score   support

    negative      0.578     0.612     0.595       428
     neutral      0.280     0.306     0.292       229
    positive      0.651     0.583     0.615       444

    accuracy                          0.537      1101
   macro avg      0.503     0.500     0.501      1101
weighted avg      0.546     0.537     0.540      1101

Assessment dataset 2
              precision    recall  f1-score   support

    negative      0.283     0.480     0.356       565
     neutral      0.450     0.310     0.367      1019
    positive      0.383     0.346     0.364       777

    accuracy                          0.363      2361
   macro avg      0.372     0.379     0.362      2361
weighted avg      0.388     0.363     0.363      2361

Mean of macro-F1 scores: 0.432


## Error analysis

This section begins to build an error-analysis framework using the dicts returned by `sst.experiment`. These have the following structure:

```
'model': trained model
'phi': the feature function used
'train_dataset':
   'X': feature matrix
   'y': list of labels
   'vectorizer': DictVectorizer,
   'raw_examples': list of raw inputs, before featurizing   
'assess_datasets': list of datasets, each with the same structure as the value of 'train_dataset'
'predictions': list of lists of predictions on the assessment datasets
'metric': `score_func.__name__`, where `score_func` is an `sst.experiment` argument
'score': the `score_func` score on the each of the assessment dataasets
```
The following function just finds mistakes, and returns a `pd.DataFrame` for easy subsequent processing:

In [ ]:
def find_errors(experiment):
    """Find mistaken predictions.

    Parameters
    ----------
    experiment : dict
        As returned by `sst.experiment`.

    Returns
    -------
    pd.DataFrame

    """
    dfs = []
    for i, dataset in enumerate(experiment['assess_datasets']):
        df = pd.DataFrame({
            'raw_examples': dataset['raw_examples'],
            'predicted': experiment['predictions'][i],
            'gold': dataset['y']})
        df['correct'] = df['predicted'] == df['gold']
        df['dataset'] = i
        dfs.append(df)
    return pd.concat(dfs)

In [ ]:
softmax_analysis = find_errors(softmax_experiment)

In [ ]:
rnn_analysis = find_errors(rnn_experiment)

Here we merge the sotmax and RNN experiments into a single DataFrame:

In [ ]:
analysis = softmax_analysis.merge(
    rnn_analysis, left_on='raw_examples', right_on='raw_examples')

analysis = analysis.drop('gold_y', axis=1).rename(columns={'gold_x': 'gold'})

The following code collects a specific subset of examples; small modifications to its structure will give you different interesting subsets:

In [ ]:
# Examples where the softmax model is correct, the RNN is not,
# and the gold label is 'positive'

error_group = analysis[
    (analysis['predicted_x'] == analysis['gold'])
    &
    (analysis['predicted_y'] != analysis['gold'])
    &
    (analysis['gold'] == 'positive')
]

In [ ]:
error_group.shape[0]

230

In [ ]:
for ex in error_group['raw_examples'].sample(5, random_state=1):
    print("="*70)
    print(ex)

Atom Egoyan has conjured up a multilayered work that tackles any number of fascinating issues
The best meal was the chicken & waffles.
My party of 5 couldn't finish the cake and took a large portion home with us (which got devoured the next day).
Pay a fraction more and go to Mortons, they'll stand on their heads to make your experience a great one!
We like to eat at the Palm 6-10 times per year and are big fans.


## Homework questions

Please embed your homework responses in this notebook, and do not delete any cells from the notebook. (You are free to add as many cells as you like as part of your responses.)

### Token-level differences [1 point]

We can begin to get a sense for how our two dev sets differ by considering the most frequent tokens from each. This question asks you to begin such analysis.

Your task: write a function `get_token_counts` that, given a `pd.DataFrame` in the format of our datasets, tokenizes the example sentences based on whitespace and creates a count distribution over all of the tokens. The function should return a `pd.Series` sorted by frequency; if you create a count dictionary `d`, then `pd.Series(d).sort_values(ascending=False)` will give you what you need.

In [ ]:
df = pd.DataFrame([
        {'sentence': 'a a b'},
        {'sentence': 'a b a'},
        {'sentence': 'a a a b.'}])

sum([Counter(item.split(" ")) for item in df["sentence"].values], Counter())

Counter({'a': 7, 'b': 2, 'b.': 1})

In [ ]:
def get_token_counts(df):

    ##### YOUR CODE HERE

    counts = sum([Counter(item.split(" ")) for item in df["sentence"].values], Counter())
    return pd.Series(counts).sort_values(ascending=False)



In [ ]:
def test_get_token_counts(func):
    df = pd.DataFrame([
        {'sentence': 'a a b'},
        {'sentence': 'a b a'},
        {'sentence': 'a a a b.'}])
    result = func(df)
    for token, expected in (('a', 7), ('b', 2), ('b.', 1)):
        actual = result.loc[token]
        assert actual == expected, \
            "For token {}, expected {}; got {}".format(
            token, expected, actual)

In [ ]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_get_token_counts(get_token_counts)

As you develop your original system, you might review these results. The two dev sets have different vocabularies and different low-level encoding details that are sure to impact model performance, especially when one considers that the train set is like `sst_dev` in all these respects. For additional discussion, see [this notebook section](sst_01_overview.ipynb#Tokenization).

### Training on some of the bakeoff data [1 point]

We have so far presented the bakeoff dev set as purely for evaluation. Since the train set consists entirely of SST-3 data, this makes the bakeoff split especially challenging. We might be able to reduce the challenging by adding some of the bakeoff dev set to the train set, keeping some of it for evaluation. The current question asks to begin explore the effects of such training.

Your task: write a function `run_mixed_training_experiment`. The function should:

1. Take as inputs (a) a model training wrapper like `fit_softmax_classifier` and (b) an integer `bakeoff_train_size` specifying the number of examples from `bakeoff_dev` that should be included in the train set.
1. Split `bakeoff_dev` so that the first `bakeoff_train_size` examples are in the train set and the rest are used for evaluation.
1. Use `sst.experiment` with the user-supplied model training wrapper, `unigrams_phi` as defined above, and a train set that consists of SST-3 train and the train portion of `bakeoff_dev` as defined in step 2. The value of `assess_dataframes` should be a list consisting of the SST-3 dev set and the evaluation portion of `bakeoff_dev` as defined in step 2.
1. Return the return value of `sst.experiment`.

The function `test_run_mixed_training_experiment` will help you iterate to the required design.

In [ ]:
def run_mixed_training_experiment(wrapper_func, bakeoff_train_size):

    ##### YOUR CODE HERE

    bakeoff_dev_train, bakeoff_dev_eval = bakeoff_dev[:bakeoff_train_size], bakeoff_dev[bakeoff_train_size:]

    experiment = sst.experiment(
    [sst_train, bakeoff_dev_train],
    unigrams_phi,
    wrapper_func,
    vectorize=True,  # For deep learning, use `vectorize=False`.
    assess_dataframes=[sst_dev, bakeoff_dev_eval])

    return experiment



In [ ]:
def test_run_mixed_training_experiment(func):
    bakeoff_train_size = 1000
    experiment = func(fit_softmax_classifier, bakeoff_train_size)

    assess_size = len(experiment['assess_datasets'])
    assert len(experiment['assess_datasets']) == 2, \
        ("The evaluation should be done on two datasets: "
         "SST3 and part of the bakeoff dev set. "
         "You have {} datasets.".format(assess_size))

    bakeoff_test_size = bakeoff_dev.shape[0] - bakeoff_train_size
    expected_eval_examples = bakeoff_test_size + sst_dev.shape[0]
    eval_examples = sum(len(d['raw_examples']) for d in experiment['assess_datasets'])
    assert expected_eval_examples == eval_examples, \
        "Expected {} evaluation examples; got {}".format(
        expected_eval_examples, eval_examples)

In [ ]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_run_mixed_training_experiment(run_mixed_training_experiment)

Assessment dataset 1
              precision    recall  f1-score   support

    negative      0.627     0.671     0.648       428
     neutral      0.319     0.162     0.214       229
    positive      0.638     0.757     0.692       444

    accuracy                          0.599      1101
   macro avg      0.528     0.530     0.518      1101
weighted avg      0.567     0.599     0.576      1101

Assessment dataset 2
              precision    recall  f1-score   support

    negative      0.471     0.412     0.440       320
     neutral      0.588     0.590     0.589       612
    positive      0.503     0.548     0.525       429

    accuracy                          0.535      1361
   macro avg      0.521     0.517     0.518      1361
weighted avg      0.534     0.535     0.534      1361

Mean of macro-F1 scores: 0.518


### A more powerful vector-averaging baseline [2 points]

In [Distributed representations as features](sst_03_neural_networks.ipynb#Distributed-representations-as-features), we looked at a baseline for the ternary SST-3 problem in which each example is modeled as the mean of its GloVe representations. A `LogisticRegression` model was used for prediction. A neural network might do better with these representations, since there might be complex relationships between the input feature dimensions that a linear classifier can't learn. To address this question, we want to get set up to run the experiment with a shallow neural classifier.

Your task: write and submit a model wrapper function around `TorchShallowNeuralClassifier`. This function should implement hyperparameter search according to this specification:

* Set `early_stopping=True` for all experiments.
* Using 3-fold cross-validation, exhaustively explore this set of hyperparameter combinations:
  * The hidden dimensionality at 50, 100, and 200.
  * The hidden activation function as `nn.Tanh()` and `nn.ReLU()`.
* For all other parameters to `TorchShallowNeuralClassifier`, use the defaults.

See [this notebook section](sst_02_hand_built_features.ipynb#Hyperparameter-search) for examples. You are not required to run a full evaluation with this function using `sst.experiment`, but we assume you will want to.

We're not evaluating the quality of your model. (We've specified the protocols completely, but there will still be variation in the results.) However, the primary goal of this question is to get you thinking more about this strong baseline feature representation scheme for SST-3, so we're sort of hoping you feel compelled to try out variations on your own.

In [ ]:
from torch_shallow_neural_classifier import TorchShallowNeuralClassifier

def fit_shallow_neural_classifier_with_hyperparameter_search(X, y):

    ##### YOUR CODE HERE
    basemod = TorchShallowNeuralClassifier(
        early_stopping=True
    )
    cv = 3
    param_grid = {
        'hidden_dim': [50, 100, 200],
        'hidden_activation': [nn.Tanh(), nn.ReLU()]
    }
    bestmod = utils.fit_classifier_with_hyperparameter_search(
        X, y, basemod, cv, param_grid)

    return bestmod

In [ ]:
# Testing the implementation
DATA_HOME = 'data'

GLOVE_HOME = os.path.join(DATA_HOME, 'glove.6B')

glove_lookup = utils.glove2dict(
    os.path.join(GLOVE_HOME, 'glove.6B.300d.txt'))

def vsm_phi(text, lookup, np_func=np.mean):
    """Represent `text` as a combination of the vector of its words.

    Parameters
    ----------
    text : str

    lookup : dict
        From words to vectors.

    np_func : function (default: np.sum)
        A numpy matrix operation that can be applied columnwise,
        like `np.mean`, `np.sum`, or `np.prod`. The requirement is that
        the function take `axis=0` as one of its arguments (to ensure
        columnwise combination) and that it return a vector of a
        fixed length, no matter what the size of the text is.

    Returns
    -------
    np.array, dimension `X.shape[1]`

    """
    allvecs = np.array([lookup[w] for w in text.split() if w in lookup])
    if len(allvecs) == 0:
        dim = len(next(iter(lookup.values())))
        feats = np.zeros(dim)
    else:
        feats = np_func(allvecs, axis=0)
    return feats

def glove_phi(text, np_func=np.mean):
    return vsm_phi(text, glove_lookup, np_func=np_func)



bestmod = sst.experiment(
    sst_train,
    glove_phi,
    fit_shallow_neural_classifier_with_hyperparameter_search,
    assess_dataframes=[sst_dev],
    vectorize=False
)

Stopping after epoch 51. Validation score did not improve by tol=1e-05 for more than 10 epochs. Final error is 5.5481029748916635

Best params: {'hidden_activation': ReLU(), 'hidden_dim': 200}
Best score: 0.524
              precision    recall  f1-score   support

    negative      0.631     0.682     0.655       428
     neutral      0.387     0.201     0.264       229
    positive      0.644     0.752     0.694       444

    accuracy                          0.610      1101
   macro avg      0.554     0.545     0.538      1101
weighted avg      0.585     0.610     0.590      1101



### BERT encoding [2 points]

We might hypothesize that encoding our examples with BERT will yield improvements over the GloVe averaging method explored in the previous question, since BERT implements a much more complex and data-driven function for this kind of combination. This question asks you to begin exploring this general hypothesis.

Your task: write a function `hf_cls_phi` that uses Hugging Face functionality to encode individual examples with BERT and returns the final output representation above the [CLS] token.

You are not required to evaluate this feature function, but it is easy to do so with `sst.experiment` and `vectorize=False` (since your feature function directly encodes every example as a vector). Your code should also be a natural basis for even more powerful approaches – for example, it might be even better to pool all the output states rather than using just the first output state. Another option is [fine-tuning](finetuning.ipynb).

In [ ]:
from transformers import BertModel, BertTokenizer
import vsm

# Instantiate a Bert model and tokenizer based on `bert_weights_name`:
bert_weights_name = 'bert-base-uncased'
##### YOUR CODE HERE
bert_tokenizer = BertTokenizer.from_pretrained(bert_weights_name)
bert_model = BertModel.from_pretrained(bert_weights_name)

def hf_cls_phi(text):
    # Get the ids. `vsm.hf_encode` will help; be sure to
    # set `add_special_tokens=True`.
    ##### YOUR CODE HERE
    encoding = vsm.hf_encode(text=text, tokenizer=bert_tokenizer, add_special_tokens=True)

    # Get the BERT representations. `vsm.hf_represent` will help:
    ##### YOUR CODE HERE
    reps = vsm.hf_represent(batch_ids=encoding, model=bert_model, layer=-1)


    # Index into `reps` to get the representation above [CLS].
    # The shape of `reps` should be (1, n, 768), where n is the
    # number of tokens. You need the 0th element of the 2nd dim:
    ##### YOUR CODE HERE
    cls_rep = reps[:,0,:].squeeze()

    # These conversions should ensure that you can work with the
    # representations flexibly. Feel free to change the variable
    # name:
    return cls_rep.cpu().numpy()




Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
def test_hf_cls_phi(func):
    rep = func("Just testing!")

    expected_shape = (768,)
    result_shape = rep.shape
    assert rep.shape == (768,), \
        "Expected shape {}; got {}".format(
        expected_shape, result_shape)

    # String conversion to avoid precision errors:
    expected_first_val = str(0.1709)
    result_first_val = "{0:.04f}".format(rep[0])

    assert expected_first_val == result_first_val, \
        ("Unexpected representation values. Expected the "
        "first value to be {}; got {}".format(
            expected_first_val, result_first_val))




In [ ]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_hf_cls_phi(hf_cls_phi)


Note: encoding all of SST-3 train (no subtrees) takes about 11 minutes on my 2015 iMac, CPU only (32GB).

In [ ]:
bestmod = sst.experiment(
    sst_train,
    hf_cls_phi,
    fit_shallow_neural_classifier_with_hyperparameter_search,
    assess_dataframes=[sst_dev],
    vectorize=False
)

Stopping after epoch 32. Validation score did not improve by tol=1e-05 for more than 10 epochs. Final error is 4.9931718707084666

Best params: {'hidden_activation': Tanh(), 'hidden_dim': 100}
Best score: 0.605
              precision    recall  f1-score   support

    negative      0.734     0.703     0.718       428
     neutral      0.381     0.188     0.251       229
    positive      0.680     0.885     0.769       444

    accuracy                          0.669      1101
   macro avg      0.598     0.592     0.580      1101
weighted avg      0.639     0.669     0.642      1101



### Your original system [3 points]

Your task is to develop an original model for the SST-3 problem and our new bakeoff dataset. There are many options. If you spend more than a few hours on this homework problem, you should consider letting it grow into your final project! Here are some relatively manageable ideas that you might try:

1. We didn't systematically evaluate the `bidirectional` option to the `TorchRNNClassifier`. Similarly, that model could be tweaked to allow multiple LSTM layers (at present there is only one), and you could try adding layers to the classifier portion of the model as well.

1. We've already glimpsed the power of rich initial word representations, and later in the course we'll see that smart initialization usually leads to a performance gain in NLP, so you could perhaps achieve a winning entry with a simple model that starts in a great place.

1. Our [practical introduction to contextual word representations](finetuning.ipynb) covers pretrained representations and interfaces that are likely to boost the performance of any system.

We want to emphasize that this needs to be an __original__ system. It doesn't suffice to download code from the Web, retrain, and submit. You can build on others' code, but you have to do something new and meaningful with it. See the course website for additional guidance on how original systems will be evaluated.

In the cell below, please provide a brief technical description of your original system, so that the teaching team can gain an understanding of what it does. This will help us to understand your code and analyze all the submissions to identify patterns and strategies.  We also ask that you report the best score your system got during development (your best average of macro-F1 scores), just to help us understand how systems performed overall.

<font color='red'>Please review the descriptions in the following comment and follow the instructions.</font>

In [ ]:
# PLEASE MAKE SURE TO INCLUDE THE FOLLOWING BETWEEN THE START AND STOP COMMENTS:
#   1) Textual description of your system.
#   2) The code for your original system.
#   3) The score achieved by your system in place of MY_NUMBER.
#        With no other changes to that line.
#        You should report your score as a decimal value <=1.0
# PLEASE MAKE SURE NOT TO DELETE OR EDIT THE START AND STOP COMMENTS

# NOTE: MODULES, CODE AND DATASETS REQUIRED FOR YOUR ORIGINAL SYSTEM
# SHOULD BE ADDED BELOW THE 'IS_GRADESCOPE_ENV' CHECK CONDITION. DOING
# SO ABOVE THE CHECK MAY CAUSE THE AUTOGRADER TO FAIL.

# START COMMENT: Enter your system description in this cell.
# My peak score was: 0.619
if 'IS_GRADESCOPE_ENV' not in os.environ:
    """This solution experiments with using TorchRNNClassifier with pretrained contexual word representations
    of text tokens based on BERT base model.

    The SST3 and Bakeoff dev datasets are used for training the system. A hyperparameter search is performed using
    SST3-train and part of bakeoff data and assessment of the best model is performed by using SST3-dev and rest
    of the bakeoff data. The following parameters are considered for the best hyperparameter search:
    - hidden_dim: Dimensionality of the RNN hidden layer
    - bidirectional: Unidirectional / bidirectional nature of the RNN model
    - num_layers: Number of layers in the RNN model
    - classifier_activation: Activation function of the classifier model

    The following are the best hyperparameters after doing the cross validation with three stratified shuffle
    splits:
    {'bidirectional': True, 'classifier_activation': ReLU(), 'hidden_dim': 200, 'num_layers': 2}.

    The final model is trained using all the data, i.e., SST3-train, SST3-dev and Bakeoff dev.
    """
    ##### YOUR CODE HERE

    # Imports
    from torch_rnn_classifier import TorchRNNClassifier
    from transformers import BertModel, BertTokenizer
    import vsm
    import sst
    import utils

    # Datasets
    bakeoff_dev = sst.bakeoff_dev_reader(SST_HOME) # Sentences from resturant reviews
    sst_train = sst.train_reader(SST_HOME) # Sentences from movie reviews
    sst_dev = sst.dev_reader(SST_HOME)


    # Best RNN classifier using bert embeddings
    bert_weights_name = 'bert-base-uncased'

    bert_tokenizer = BertTokenizer.from_pretrained(bert_weights_name)
    bert_model = BertModel.from_pretrained(bert_weights_name)

    def bert_phi(text):
        """Get the BERT last hidden state embeddings text tokens.
        """
        encoding = vsm.hf_encode(text=text, tokenizer=bert_tokenizer, add_special_tokens=True)
        reps = vsm.hf_represent(batch_ids=encoding, model=bert_model, layer=-1)
        return reps.squeeze(0).numpy()


    def system_RNN_bert(X, y):
        """Gridsearch on different hyperparameters of an RNN model using token embeddings based on BERT base
        model.
        """
        basemod = TorchRNNClassifier(
            vocab = [],
            early_stopping=True,
            use_embedding=False
        )
        cv = 3
        ## Add the bidirectional and num_layers in the grid search parameters in
        ## compariosn to typically done in course
        param_grid = {
            'hidden_dim': [100, 200],
            'classifier_activation': [nn.Tanh(), nn.ReLU()],
            'bidirectional': [True, False],
            'num_layers': [1, 2]
        }

        bestmod = utils.fit_classifier_with_hyperparameter_search(
            X, y, basemod, cv, param_grid)

        return bestmod

    # Use part of the bake off dev set for training so that model can learn the trends specific to bakeoff dataset
    bakeoff_train_size = bakeoff_dev.shape[0] // 2
    bakeoff_dev_train, bakeoff_dev_eval = bakeoff_dev[:bakeoff_train_size], bakeoff_dev[bakeoff_train_size:]

    bestmod_bert = sst.experiment(
        [sst_train, bakeoff_dev_train],
        bert_phi,
        system_RNN_bert,
        assess_dataframes=[sst_dev, bakeoff_dev_eval],
        vectorize=False
    )

    # Train the best model on all the data.

    # Train the best model on complete dataset
    train = sst.build_dataset(
            [sst_train, sst_dev, bakeoff_dev],
            bert_phi,
            vectorizer=None,
            vectorize=False)

    X_train = train['X']
    y_train = train['y']

    bestmod_bert["model"] = bestmod_bert["model"].fit(X_train, y_train)

# STOP COMMENT: Please do not remove this comment.

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Stopping after epoch 22. Validation score did not improve by tol=1e-05 for more than 10 epochs. Final error is 0.1123042604885995465

Best params: {'bidirectional': True, 'classifier_activation': ReLU(), 'hidden_dim': 100, 'num_layers': 2}
Best score: 0.636
Assessment dataset 1
              precision    recall  f1-score   support

    negative      0.739     0.650     0.692       428
     neutral      0.343     0.310     0.326       229
    positive      0.703     0.820     0.757       444

    accuracy                          0.648      1101
   macro avg      0.595     0.593     0.591      1101
weighted avg      0.642     0.648     0.642      1101

Assessment dataset 2
              precision    recall  f1-score   support

    negative      0.623     0.550     0.584       280
     neutral      0.677     0.664     0.671       524
    positive      0.652     0.727     0.688       377

    accuracy                          0.657      1181
   macro avg      0.651     0.647     0.648      1181
weighted avg      0.656     0.657     0.656      1181

Mean of macro-F1 scores: 0.619


Stopping after epoch 32. Validation score did not improve by tol=1e-05 for more than 10 epochs. Final error is 0.006055449310224503

## Bakeoff [1 point]

As we said above, the bakeoff evaluation data is the official SST test-set release and a new test set derived from the same sources and labeling methods as for `bakeoff_dev`.

For this bakeoff, you'll evaluate your original system from the above homework problem on these test sets. Our metric will be the mean of the macro-F1 values, which weights both datasets equally despite their differing sizes.

The central requirement for your system is that you have define a `predict_one` method for it that maps a text (str) directly to a label prediction – one of 'positive', 'negative', 'neutral'. If you used `sst.experiment` with `vectorize=True`, then the following function (for `softmax_experiment`) will be easy to adapt – you probably just need to change the variable `softmax_experiment` to the variable for your experiment output.

In [ ]:
def predict_one_softmax(text):
    # Singleton list of feature dicts:
    feats = [softmax_experiment['phi'](text)]
    # Vectorize to get a feature matrix:
    X = softmax_experiment['train_dataset']['vectorizer'].transform(feats)
    # Standard sklearn `predict` step:
    preds = softmax_experiment['model'].predict(X)
    # Be sure to return the only member of the predictions,
    # rather than the singleton list:
    return preds[0]


If you used an RNN like the one we demoed above, then featurization is a bit more straightforward:

In [ ]:
def predict_one_rnn(text):
    # List of tokenized examples:
    X = [bestmod_bert['phi'](text)]
    # Standard `predict` step on a list of lists of str:
    preds = bestmod_bert['model'].predict(X)
    # Be sure to return the only member of the predictions,
    # rather than the singleton list:
    return preds[0]



The following function is used to create the bakeoff submission file. Its arguments are your `predict_one` function and an output filename (str).

In [2]:
import os

if 'IS_GRADESCOPE_ENV' not in os.environ:
    create_bakeoff_submission(predict_one_rnn)

NameError: name 'create_bakeoff_submission' is not defined

In [ ]:
def create_bakeoff_submission(
        predict_one_func,
        output_filename='cs224u-sentiment-bakeoff-entry.csv'):

    bakeoff_test = sst.bakeoff_test_reader(SST_HOME)
    sst_test = sst.test_reader(SST_HOME)
    bakeoff_test['dataset'] = 'bakeoff'
    sst_test['dataset'] = 'sst3'
    df = pd.concat((bakeoff_test, sst_test))

    df['prediction'] = df['sentence'].apply(predict_one_func)

    df.to_csv(output_filename, index=None)



Thus, for example, the following will create a bake-off entry based on `predict_one_softmax`:

In [ ]:
# This check ensure that the following code only runs on the local environment only.
# The following call will not be run on the autograder environment.
if 'IS_GRADESCOPE_ENV' not in os.environ:
    create_bakeoff_submission(predict_one_rnn)

